# Attribute 5 - Relativity Affordability

#### Rationale
Be on the cheapest land within the Reconstruction Envelope, e.g. harvested plantation forest will likely be cheaper land than lowland dairy farmland. Note that any natural land cover types will be excluded because by default they will not be part of a reconstruction envelope

#### Data layers
Landcover database link

## Scoring
- 1.0: Depleted grassland, Urban parkland/ open space
- 0.8: Deciduous Hardwoods (dryland only), Forest-harvested
- 0.6: Low Producing Grassland, Short-rotation cropland
- 0.4: Plantation forest
- 0.2: High Producing Grassland, Orchard, Vineyard or Other Perennial Crop
- 0.0: Everything else

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import math
import os
import time
from os import listdir

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pyproj
import rasterio
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import from_origin
from shapely.geometry import Point, Polygon, box

from constants import small_polygon_threshold, m2_to_ha, x_resolution, y_resolution, keep_cols, keep_cols_catch

# Load

In [3]:
%%time
lcdb = gpd.read_file('../BaseLayersExternal/lcdb-v5/lcdb-v50-land-cover-database-version-50-mainland-new-zealand.shp')
lcdb = lcdb.to_crs('epsg:2193')


CPU times: total: 31.6 s
Wall time: 31.8 s


In [4]:
%%time
# Area of Interest
aoi = gpd.read_file("../BaseLayersEco-index/Eco-index_RestorableAreas__Catchments_v290824.gpkg")
aoi = aoi[['Catchment', 'geometry']].copy()
aoi.sindex
aoi.shape

CPU times: total: 1.45 s
Wall time: 2.68 s


(45989, 2)

In [5]:
lcs = gpd.read_file('../BaseLayersEco-index/Eco-index_LandCoverSnapshot_v290824.gpkg')
lcs.LCDBLandCoverClass.value_counts()

LCDBLandCoverClass
Exotic Forest                                97016
Indigenous Forest                            56886
Manuka and/or Kanuka                         53755
Broadleaved Indigenous Hardwoods             46786
High Producing Exotic Grassland              33555
Gravel or Rock                               30452
Low Producing Grassland                      27554
Deciduous Hardwoods                          18935
Gorse and/or Broom                           18225
Sub Alpine Shrubland                         12951
Lake or Pond                                 11577
Tall Tussock Grassland                       11490
Herbaceous Freshwater Vegetation             11089
Forest - Harvested                            8456
Built-up Area (settlement)                    8173
Matagouri or Grey Scrub                       8136
Orchard, Vineyard or Other Perennial Crop     6173
Landslide                                     6059
Alpine Grass/Herbfield                        5897
Mixed Exotic

In [6]:
cover_mapping = {
'Depleted Grassland': 10,
'Urban Parkland/Open Space': 10,
'Forest - Harvested': 8,
'Low Producing Grassland': 6,
'Short-rotation Cropland': 6,
'Exotic Forest': 4,
'High Producing Exotic Grassland': 2,
'Orchard, Vineyard or Other Perennial Crop': 2,
# cover_mapping['Deciduous Hardwoods'] Handled later
}

lcs['PixelScore'] = lcs.LCDBLandCoverClass.map(cover_mapping, na_action='ignore').fillna(0)
lcs.loc[(lcs['LCDBLandCoverClass'] == 'Deciduous Hardwoods')
             & (lcs['LCDBWetlandContext']=='no'), 'PixelScore'] = 8
lcs['PixelDesc'] = lcs.LCDBLandCoverClass.copy()
lcs['PrioOption'] = 'Relative Affordability'



In [7]:
cost = lcs.copy()
cost[~pd.isna(lcs.PixelScore)].copy().reset_index(drop=True)
cost.sindex

In [8]:
cost['PixelScore'].value_counts()

PixelScore
0.0     309364
4.0      97016
2.0      39728
6.0      31922
8.0      26248
10.0      6812
Name: count, dtype: int64

# Calculate Intersection with All Catchments

In [9]:
%%time
aoi_catch = aoi.overlay(cost)

CPU times: total: 36min 35s
Wall time: 36min 49s


C:\Users\dav\miniconda3_9\envs\eco\Lib\site-packages\geopandas\geodataframe.py:2675: UserWarning: `keep_geom_type=True` in overlay resulted in 530363 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  return geopandas.overlay(


In [10]:
aoi_catch['Area_ha'] = aoi_catch.area * m2_to_ha
aoi_catch = aoi_catch[aoi_catch.Area_ha > small_polygon_threshold].reset_index(drop=True)
aoi_catch['Area_ha'] = aoi_catch['Area_ha'].round(2)

aoi_catch[keep_cols_catch].to_file(f"../OutputArtifacts/A05_RelativeAffordability/A05_RelativeAfforability_20240829.gpkg")

## Save each catchment for future reference

In [11]:
for n_catch, catch in enumerate(aoi.Catchment.sort_values().unique()):
    start_time = time.time()
    current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(start_time))
    print(f"\n{n_catch}_{catch.upper()}      {current_time}")

    ## Filter to each catchment
    sub_aoi_catch = aoi_catch[aoi_catch['Catchment'] == catch][keep_cols_catch].reset_index(drop=True)

    if sub_aoi_catch.shape[0] > 0:
        # Add Area_ha column
        base_path = r'..\OutputArtifacts\A05_RelativeAffordability\A05_Catchments'
        sub_aoi_catch.to_file(f"{base_path}\{str(n_catch).zfill(3)}_{catch}_RelativeAfforability_20240829.gpkg")

    end_time = time.time()
    time_diff = end_time - start_time
    formatted_time_diff = time.strftime("%M:%S", time.gmtime(time_diff))
    print(f"    {catch} Saved. Elapsed time: {formatted_time_diff}")


0_APARIMA      2024-09-16 15:43:29
    Aparima Saved. Elapsed time: 00:00

1_ASHBURTON-HINDS      2024-09-16 15:43:29
    Ashburton-Hinds Saved. Elapsed time: 00:00

2_AUCKLAND BASIN      2024-09-16 15:43:29
    Auckland Basin Saved. Elapsed time: 00:00

3_AUCKLAND OFFSHORE ISLANDS      2024-09-16 15:43:29
    Auckland offshore islands Saved. Elapsed time: 00:00

4_AWATERE      2024-09-16 15:43:29
    Awatere Saved. Elapsed time: 00:00

5_BANKS PENINSULA      2024-09-16 15:43:29
    Banks Peninsula Saved. Elapsed time: 00:00

6_BAY OF PLENTY OFFSHORE ISLANDS      2024-09-16 15:43:30
    Bay of Plenty offshore islands Saved. Elapsed time: 00:00

7_BLUFF OFFSHORE      2024-09-16 15:43:30
    Bluff offshore Saved. Elapsed time: 00:00

8_BULLER      2024-09-16 15:43:30
    Buller Saved. Elapsed time: 00:00

9_CATLINS      2024-09-16 15:43:30
    Catlins Saved. Elapsed time: 00:00

10_CLARENCE      2024-09-16 15:43:30
    Clarence Saved. Elapsed time: 00:00

11_CLUTHA      2024-09-16 15:43

# Calculate Intersection Catchment by Catchment

In [12]:
# import time

# sub_dfs = {}

# keep_cols = ['Attribute', 'PixelScore', 'PixelDesc', 'Area_ha', 'geometry']

# for n_catch, catch in enumerate(aoi.Catchment.sort_values().unique()):

#     current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(time.time()))
#     print(f'\n{catch.upper()}\n{current_time}')

#     aoi_sub = aoi[aoi.Catchment==catch].copy()
    
#     aoi_bounds = aoi_sub.total_bounds  # Returns [minx, miny, maxx, maxy]

#     lcdb_filtered = lcdb.cx[aoi_bounds[0]:aoi_bounds[2], aoi_bounds[1]:aoi_bounds[3]]

#     fig, ax = plt.subplots(figsize=(12,5))
#     lcdb_filtered.plot(ax=ax)
#     aoi_sub.plot(ax=ax, color='red')
#     plt.show()   

#     sub_lcdb_env = lcdb_filtered.overlay(aoi_sub, how='intersection')

#     ## Filter na and Small Areas
#     sub_lcdb_env['Area_ha'] = sub_lcdb_env.area * m2_to_ha

#     sub_lcdb_env = sub_lcdb_env[sub_lcdb_env.Area_ha > 0.005]
#     sub_lcdb_env[keep_cols].to_file(f"output_layers/attr05/attr05_by_catchment/{str(n_catch).zfill(3)}_{catch}_lcdb_env.shp")
#     sub_dfs[catch] = sub_lcdb_env[keep_cols]